# 01_setup — Structure Unity Catalog et dépôt des données
**Projet :** Alpenia Lakehouse · Niveau 1 · Couche bronze

Ce notebook crée la hiérarchie Unity Catalog **catalogue → schéma → volume** et vérifie que les dix fichiers sources ont bien été déposés dans le volume `landing`.

Il est **ré-exécutable** : toutes les commandes `CREATE` utilisent `IF NOT EXISTS`, elles ne font rien si l'objet existe déjà.

> Compute : ce notebook s'attache automatiquement au **compute serverless** des notebooks (aucun cluster à créer).

## 1. Création du catalogue, du schéma et du volume

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS alpenia_dev
COMMENT 'Alpenia Assurances - environnement de développement';

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS alpenia_dev.bronze
COMMENT 'Couche brute : une table Delta par source, données telles que reçues puis typées';

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS alpenia_dev.bronze.landing
COMMENT 'Zone de dépôt des exports bruts des systèmes sources (CSV, JSON, Parquet)';

In [ ]:
%sql
-- Vérification : le schéma et le volume existent
SHOW SCHEMAS IN alpenia_dev;

In [ ]:
%sql
SHOW VOLUMES IN alpenia_dev.bronze;

## 2. Dépôt des fichiers

À faire **à la main** dans le Catalog Explorer : `alpenia_dev` → `bronze` → `landing` → *Upload to this volume*.
Créer un dossier par source puis y téléverser les fichiers, en respectant exactement cette arborescence :

```
landing/
├── contracts/    contracts.csv
├── clients/      clients.csv
├── pricing/      pricing.csv
├── claims/       claims_2026_01.parquet, claims_2026_02.parquet, claims_2026_03.parquet
├── web_events/   web_events_2026-03-24_10h.json, _11h.json, _12h.json
└── expertises/   expertises.json
```

## 3. Contrôle de l'arborescence et des volumétries
Un dossier mal nommé ferait échouer silencieusement `02_ingestion` (DataFrame vide). On vérifie donc dossier par dossier le nombre de fichiers attendu et leur taille.

In [ ]:
BASE = "/Volumes/alpenia_dev/bronze/landing"

# dossier -> nombre de fichiers attendus
EXPECTED_FILES = {
    "contracts": 1,
    "clients": 1,
    "pricing": 1,
    "claims": 3,
    "web_events": 3,
    "expertises": 1,
}

rows = []
for folder, expected in EXPECTED_FILES.items():
    try:
        files = [f for f in dbutils.fs.ls(f"{BASE}/{folder}/") if not f.name.endswith("/")]
    except Exception:
        files = []   # dossier absent ou mal nommé
    for f in files:
        rows.append((folder, f.name, f.size, round(f.size / 1024, 1)))
    status = "OK" if len(files) == expected else "ERREUR"
    print(f"{folder:<12} {len(files)}/{expected} fichier(s)  {status}")

df_files = spark.createDataFrame(rows, "folder STRING, file_name STRING, size_bytes LONG, size_kb DOUBLE")
display(df_files.orderBy("folder", "file_name"))

In [ ]:
# Garde-fou : on stoppe ici si l'arborescence n'est pas conforme
total = df_files.count()
assert total == 10, f"{total} fichier(s) trouvé(s) au lieu de 10 : vérifier l'arborescence du volume landing"
print(f"Arborescence conforme : {total} fichiers, {df_files.groupBy().sum('size_bytes').first()[0]:,} octets au total")

In [ ]:
# Aucun fichier ne doit traîner à la racine du volume
root_files = [f.name for f in dbutils.fs.ls(BASE) if not f.name.endswith("/")]
print("Fichiers à la racine :", root_files or "aucun (OK)")